# Preparing Annual Corn Prices

This notebook loads USDA NASS annual corn price data (1866-2025), adjusts prices for inflation using the Consumer Price Index (CPI), and prepares the data for predictive modeling.

## Data Loading

Import libraries and load the corn price and CPI datasets from the project's GitHub repository.

In [20]:
# Import data manipulation libraries
import pandas as pd
import numpy as np

# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Load data from GitHub repository
base_url = "https://raw.githubusercontent.com/chuckgrigsby0/agec-370/main/data/"
corn_prices = pd.read_csv(base_url + 'corn_prices_bushels.csv')

# Load CPI data directly from GitHub URL
cpi = pd.read_csv(base_url + 'cpi_1913_2024.csv')

## Data Cleaning

The `Value` column from NASS data often contains suppressed entries marked `(D)` or `(Z)`, comma-formatted numbers, and mixed types. The steps below standardize it into a clean numeric column.

In [21]:
value_clean = corn_prices['Value'].fillna('').astype(str)

# Remove any row containing (D) or (Z) anywhere
mask = value_clean.str.contains(r'\([DZ]\)', regex=True)
corn_prices = corn_prices[~mask].copy()

# Remove ',' from `Value` column
corn_prices['Value'] = (
    corn_prices['Value']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)
# Convert to numeric
corn_prices['Value'] = pd.to_numeric(corn_prices['Value'], errors='coerce')

# Check for NAs
print(corn_prices['Value'].isna().sum())

# Optional: Drop NAs if needed
# corn_prices.dropna(subset=['Value'], inplace=True) 

0


### Rename `Value` to `nominal_price`

Let's rename the `Value` column to `nominal_price` before we add the CPI-adjusted `real_price` column later.

In [22]:
corn_prices = corn_prices.rename(columns={'Value': 'nominal_price'})

## Inflation Adjustment

To compare prices over time, we adjust nominal prices to real (inflation-adjusted) prices using the Consumer Price Index (CPI). We will use 2024 as the base year, so all prices are in terms of 2024 dollars.

The adjustment formula is: `real_price = (nominal_price / CPI_t) * CPI_2024`

### Merge CPI Data

In [23]:
corn_prices = corn_prices.merge(
    cpi,  # CPI data loaded earlier
    left_on='year',  # Match on 'year' column in corn price data
    right_on='year',  # Match on 'Year' column in CPI data
    how='left'  # Keep all corn price records
)

### Drop Missing Values

Rows without a matching CPI year cannot be inflation-adjusted and are dropped.

In [24]:
corn_prices = corn_prices.dropna(subset=['annual_avg_cpi'])

### Set the Base Year CPI

Retrieve the 2024 CPI value. This serves as the numerator in the inflation adjustment formula, scaling all prices up to 2024 purchasing power.

In [25]:
cpi_base_year = cpi[cpi['year'] == 2024]['annual_avg_cpi'].values[0]  # Should be 313.689
print(cpi_base_year)

313.689


### Calculate Real Prices

Apply the CPI adjustment to produce `real_price` in 2024 dollars.

In [26]:
corn_prices['real_price'] = (
    corn_prices['nominal_price'] / corn_prices['annual_avg_cpi']
) * cpi_base_year

In [27]:
sel_columns = ['year', 'freq_desc', 'unit_desc', 'short_desc', 'statisticcat_desc', 'nominal_price', 'annual_avg_cpi', 'real_price']

corn_prices = corn_prices[sel_columns]

In [28]:
corn_prices.to_csv('./data/corn_prices_bushels_nominal_real.csv', index=False)